In [1]:
!pip install -q langchain langchain-google-genai langchain-community

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 855.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 19.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatib

In [8]:

import os
import sqlite3
from google.colab import userdata
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
import gradio as gr


os.environ["GOOGLE_API_KEY"]="AQ.Ab8RN6Lji7BoFClGakMepUiPnTlvaGr5Clou2pPuorbzJ9V-rw"

conn=sqlite3.connect("students.db")
cur=conn.cursor()

cur.execute("DROP TABLE IF EXISTS students")
cur.execute("""
CREATE TABLE students(
    student_id TEXT PRIMARY KEY,
    name TEXT,
    department TEXT,
    python INTEGER,
    database INTEGER,
    ai INTEGER,
    web INTEGER
)
""")

students=[
    ("22CS045","Dhanushya","Computer Science",85,72,90,78),
    ("22CS046","Rahul","Computer Science",65,70,68,72),
    ("22CS047","Priya","Information Technology",92,88,95,90),
    ("22CS048","Arun","Information Technology",55,60,58,62),
    ("22CS049","Meena","Computer Science",78,85,80,88)
]

cur.executemany("INSERT INTO students VALUES(?,?,?,?,?,?,?)",students)
conn.commit()
conn.close()

def _query(sql,params=()):
    conn=sqlite3.connect("students.db")
    row=conn.execute(sql,params).fetchone()
    conn.close()
    return row

@tool
def get_student_info(student_id:str)->str:
    """Get the name and department of a student using their student ID."""
    row=_query("SELECT name,department FROM students WHERE student_id=?",(student_id,))
    if not row:
        return f"No student found with ID {student_id}."
    return f"Name: {row[0]}, Department: {row[1]}"

@tool
def get_student_marks(student_id:str)->str:
    """Get the Python, Database, AI and Web marks of a student."""
    row=_query("SELECT python,database,ai,web FROM students WHERE student_id=?",(student_id,))
    if not row:
        return f"No marks found for ID {student_id}."
    return f"Python: {row[0]}, Database: {row[1]}, AI: {row[2]}, Web: {row[3]}"

@tool
def calculator(expression:str)->str:
    """Calculate total or average marks using a mathematical expression."""
    try:
        return str(eval(expression,{"__builtins__":{}},{}))
    except Exception as e:
        return f"Error: {e}"

@tool
def get_passing_rules()->str:
    """Return the university passing rules."""
    return("University passing rules: "
           "1) Minimum overall average = 40%. "
           "2) Minimum mark in each subject = 35%.")

llm=ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0,
    max_output_tokens=500
)

agent=create_agent(
    model=llm,
    tools=[get_student_info,get_student_marks,calculator,get_passing_rules],
    system_prompt="""
    You are a student assistant. Use the provided tools to answer questions.

    Rules:
    - For name and department questions, use get_student_info.
    - For marks questions, use get_student_marks.
    - To calculate total or average, get marks first, then use calculator.
    - To check pass eligibility, get marks, calculate the average, fetch passing rules, and compare.
    - Never repeat the same tool call with the same input.
    - Always state the final answer clearly.
    """
)

def ask(question):
    result=agent.invoke({
        "messages":[{"role":"user","content":question}]
    })

    answer=result["messages"][-1].content

    if isinstance(answer,list):
        answer="\n".join(item.get("text","") for item in answer if isinstance(item,dict))

    return answer

questions=[
    "What is the name and department of student 22CS045?",
    "What are the marks of 22CS047?",
    "What is the total and average mark of 22CS045?",
    "Is 22CS045 eligible to pass according to the university rules?",
    "I am 22CS045. Tell me my name, department, total marks, average marks, and whether I satisfy the university passing requirements."
]

for q in questions:
    print("\n"+"="*70)
    print("Q:",q)
    print("-"*70)
    print("A:",ask(q))

def chatbot(message,history):
    return ask(message)

demo=gr.ChatInterface(
    fn=chatbot,
    title="Student Assistant",
    description="Ask about students, marks, totals, averages, and pass eligibility."
)

demo.launch()



Q: What is the name and department of student 22CS045?
----------------------------------------------------------------------


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


A: The student with ID 22CS045 is Dhanushya, and their department is Computer Science.

Q: What are the marks of 22CS047?
----------------------------------------------------------------------


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


A: The marks for student 22CS047 are:
- Python: 92
- Database: 88
- AI: 95
- Web: 90

Q: What is the total and average mark of 22CS045?
----------------------------------------------------------------------


...

A: The total mark of student 22CS045 is 325, and the average mark is 81.25.

Q: Is 22CS045 eligible to pass according to the university rules?
----------------------------------------------------------------------


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


A: To determine if student 22CS045 is eligible to pass, we review their marks against the university passing rules:

1. **Individual Subject Marks:** Python (85), Database (72), AI (90), and Web (78). All marks are above the minimum required threshold of 35% in each subject.
2. **Overall Average:** The student's average is 81.25%, which is well above the minimum overall average requirement of 40%.

Therefore, student 22CS045 is **eligible to pass**.

Q: I am 22CS045. Tell me my name, department, total marks, average marks, and whether I satisfy the university passing requirements.
----------------------------------------------------------------------


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


A: **Student Information:**
- **Name:** Dhanushya
- **Department:** Computer Science

**Marks:**
- **Python:** 85
- **Database:** 72
- **AI:** 90
- **Web:** 78

**Calculations:**
- **Total Marks:** 325
- **Average Marks:** 81.25%

**Passing Requirements Check:**
- University Rules: 
  1. Minimum overall average = 40% (Your average is 81.25%, which satisfies this).
  2. Minimum mark in each subject = 35% (Your lowest mark is 72, which is above 35%, satisfying this).

**Conclusion:** 
Yes, you satisfy all university passing requirements.
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fdc2bc951780d34b14.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hostin